In [1]:
import cobra
from cobra.flux_analysis import flux_variability_analysis
from determine_trait_flux_basis import determine_trait_flux_basis
from enumerate_evolvex import enumerate_evolvex
from get_growth_supporting_niche12 import get_growth_supporting_niche12

In [2]:
model = cobra.io.read_sbml_model("iML1515.xml")

#1.Nullstiller objektivfunksjon og setter biomasse som objektiv
for rxn in model.reactions:
    rxn.objective_coefficient = 0
model.reactions.get_by_id("BIOMASS_Ec_iML1515_core_75p37M").objective_coefficient = 1

In [3]:
#2.Create a copy for flux-basis determination
model_wt = model.copy()

#3.Run FVA to define wild-type flux ranges
fva_result = flux_variability_analysis(
    model_wt,
    reaction_list=model_wt.reactions,
    fraction_of_optimum=1.0
)
wtminmax = fva_result[['minimum', 'maximum']].to_numpy() #benyttes i determine_trait_flux_basis etterpå
#WT intervall er når modellen optimaliserer vekst. 
#lb og ub er for trait-modellen med ønsket objektivfunksjon. 

In [33]:
#4.Definerer den nye objektivfunksjonen
trait_model = model.copy() #M9 medium
for rxn in trait_model.reactions:
    rxn.objective_coefficient = 0

trait=trait_model.reactions.get_by_id("EX_lac__D_e")
trait.lower_bound=0 #blokkerer opptak. Hindre at den ikke øke fluxen ved å ta opp direkte fra medium. 
trait.upper_bound=1000 
trait.objective_coefficient = 1 #setter ønsket reaksjon som objektivfunksjon

#sjekker om det støtter vekst
sol = trait_model.optimize()
print(sol.objective_value)

20.0


In [34]:
#Basemedium, det universale som ligger i miljøet
lb_open = ["EX_ala__L_e","EX_glu__L_e","EX_gln__L_e","EX_asp__L_e","EX_ser__L_e","EX_gly_e","EX_leu__L_e","EX_lys__L_e", "EX_thm_e", "EX_btn_e"]

def open_uptake(trait_model, lb_open):
    for rid in lb_open:
        if rid in trait_model.reactions:
            trait_model.reactions.get_by_id(rid).lower_bound = -abs(10)

def aapen(trait_model, threshold=-1e-9):
    open_ex = []
    for r in trait_model.reactions:
        if r.id.startswith("EX_") and r.lower_bound < threshold:
            open_ex.append((r.id, r.lower_bound, r.upper_bound))
    return sorted(open_ex)

open_uptake(trait_model, lb_open)
open_uptakes = aapen(trait_model)
for x in open_uptakes[:100]:
    print(x)

('EX_ala__L_e', -10, 1000.0)
('EX_asp__L_e', -10, 1000.0)
('EX_btn_e', -10, 1000.0)
('EX_ca2_e', -1000.0, 1000.0)
('EX_cl_e', -1000.0, 1000.0)
('EX_co2_e', -1000.0, 1000.0)
('EX_cobalt2_e', -1000.0, 1000.0)
('EX_cu2_e', -1000.0, 1000.0)
('EX_fe2_e', -1000.0, 1000.0)
('EX_fe3_e', -1000.0, 1000.0)
('EX_glc__D_e', -10.0, 1000.0)
('EX_gln__L_e', -10, 1000.0)
('EX_glu__L_e', -10, 1000.0)
('EX_gly_e', -10, 1000.0)
('EX_h2o_e', -1000.0, 1000.0)
('EX_h_e', -1000.0, 1000.0)
('EX_k_e', -1000.0, 1000.0)
('EX_leu__L_e', -10, 1000.0)
('EX_lys__L_e', -10, 1000.0)
('EX_mg2_e', -1000.0, 1000.0)
('EX_mn2_e', -1000.0, 1000.0)
('EX_mobd_e', -1000.0, 1000.0)
('EX_na1_e', -1000.0, 1000.0)
('EX_nh4_e', -1000.0, 1000.0)
('EX_ni2_e', -1000.0, 1000.0)
('EX_o2_e', -1000.0, 1000.0)
('EX_pi_e', -1000.0, 1000.0)
('EX_sel_e', -1000.0, 1000.0)
('EX_ser__L_e', -10, 1000.0)
('EX_slnt_e', -1000.0, 1000.0)
('EX_so4_e', -1000.0, 1000.0)
('EX_thm_e', -10, 1000.0)
('EX_tungs_e', -1000.0, 1000.0)
('EX_zn2_e', -1000.0, 1000.

In [35]:
#6.Determine_trait_flux_basis
import numpy as np 
targets, dirs, signs, dirSolp, binRxns, fullSolOpt, solutionp = determine_trait_flux_basis(trait_model, 'max', wtminmax, 1, 1, 0.00001, 5)
#7.parse-funksjon som fikser evolvex input
targets_flat = targets[0]
dirs_flat = dirs[0]

evolveX_targets = [int(x) for x in targets_flat]
evolveX_target_dirs = list(dirs_flat)

# lag dict til evolveX_score
targets_dict = {model.reactions[i].id: d for i, d in zip(evolveX_targets, evolveX_target_dirs)}

print(targets_dict)



{'ICDHyr': 'DOWN', 'PPCK': 'UP', 'PTAr': 'DOWN', 'ACKr': 'UP', 'PGI': 'DOWN', 'PGL': 'UP', 'RPE': 'UP', 'TALA': 'UP', 'TKT1': 'UP', 'ASPT': 'UP', 'SERD_L': 'UP', 'OAADC': 'UP', 'GLYCL': 'UP', 'CITL': 'UP', 'GHMT2r': 'DOWN', 'F6PA': 'DOWN', 'NADTRHD': 'UP', 'GND': 'UP', 'G6PDH2r': 'UP', 'GLUtex': 'UP', 'GLYtex': 'UP', 'TKT2': 'UP', 'GLNtex': 'UP', 'D_LACtex': 'DOWN', 'ASPtex': 'UP', 'ALKP': 'UP', 'SERtex': 'UP', 'SERt4pp': 'UP', 'NH4tpp': 'DOWN', 'GLUNpp': 'UP', 'NH4tex': 'DOWN', 'ALAtex': 'UP', 'H2Otpp': 'UP', 'GLUDy': 'UP', 'LDH_D': 'DOWN', 'D_LACt2pp': 'DOWN', 'TPI': 'DOWN', 'RPI': 'DOWN', 'ACONTb': 'DOWN', 'ACONTa': 'DOWN', 'ALAt2pp': 'UP', 'POR5': 'DOWN', 'FLDR2': 'UP', 'GLYtpp': 'DOWN'}


In [36]:
#8.Prepare EvolveX-like growth-constrained model
model_evolvex = model.copy()

#9.Stenger glukose og ammonium for å finne alternative ruter 
model_evolvex.reactions.get_by_id("EX_glc__D_e").lower_bound = 0
model_evolvex.reactions.get_by_id("EX_glc__D_e").upper_bound = 0

In [10]:
#12.lager kombinasjoner som støtter vekst med alternative kilder
condMap = {
    "carbon": ["EX_gal_e", "EX_fru_e", "EX_ac_e", "EX_pyr_e", "EX_succ_e", "EX_glyc_e"],
    "nitrogen": ["EX_gln__L_e", "EX_glu__L_e","EX_ala__L_e", "EX_ser__L_e", "EX_nh4_e"]
}

growth, uptakes, combos = get_growth_supporting_niche12(
    model_evolvex,
    condMap
)

print(growth, uptakes, combos)


1 EX_gal_e EX_gln__L_e EX_glu__L_e
2 EX_gal_e EX_gln__L_e EX_ala__L_e
3 EX_gal_e EX_gln__L_e EX_ser__L_e
4 EX_gal_e EX_gln__L_e EX_nh4_e
5 EX_gal_e EX_glu__L_e EX_ala__L_e
6 EX_gal_e EX_glu__L_e EX_ser__L_e
7 EX_gal_e EX_glu__L_e EX_nh4_e
8 EX_gal_e EX_ala__L_e EX_ser__L_e
9 EX_gal_e EX_ala__L_e EX_nh4_e
10 EX_gal_e EX_ser__L_e EX_nh4_e
11 EX_fru_e EX_gln__L_e EX_glu__L_e
12 EX_fru_e EX_gln__L_e EX_ala__L_e
13 EX_fru_e EX_gln__L_e EX_ser__L_e
14 EX_fru_e EX_gln__L_e EX_nh4_e
15 EX_fru_e EX_glu__L_e EX_ala__L_e
16 EX_fru_e EX_glu__L_e EX_ser__L_e
17 EX_fru_e EX_glu__L_e EX_nh4_e
18 EX_fru_e EX_ala__L_e EX_ser__L_e
19 EX_fru_e EX_ala__L_e EX_nh4_e
20 EX_fru_e EX_ser__L_e EX_nh4_e
21 EX_ac_e EX_gln__L_e EX_glu__L_e
22 EX_ac_e EX_gln__L_e EX_ala__L_e
23 EX_ac_e EX_gln__L_e EX_ser__L_e
24 EX_ac_e EX_gln__L_e EX_nh4_e


/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


25 EX_ac_e EX_glu__L_e EX_ala__L_e
26 EX_ac_e EX_glu__L_e EX_ser__L_e
27 EX_ac_e EX_glu__L_e EX_nh4_e
28 EX_ac_e EX_ala__L_e EX_ser__L_e
29 EX_ac_e EX_ala__L_e EX_nh4_e
30 EX_ac_e EX_ser__L_e EX_nh4_e
31 EX_pyr_e EX_gln__L_e EX_glu__L_e
32 EX_pyr_e EX_gln__L_e EX_ala__L_e
33 EX_pyr_e EX_gln__L_e EX_ser__L_e
34 EX_pyr_e EX_gln__L_e EX_nh4_e
35 EX_pyr_e EX_glu__L_e EX_ala__L_e
36 EX_pyr_e EX_glu__L_e EX_ser__L_e
37 EX_pyr_e EX_glu__L_e EX_nh4_e
38 EX_pyr_e EX_ala__L_e EX_ser__L_e
39 EX_pyr_e EX_ala__L_e EX_nh4_e
40 EX_pyr_e EX_ser__L_e EX_nh4_e
41 EX_succ_e EX_gln__L_e EX_glu__L_e
42 EX_succ_e EX_gln__L_e EX_ala__L_e
43 EX_succ_e EX_gln__L_e EX_ser__L_e
44 EX_succ_e EX_gln__L_e EX_nh4_e
45 EX_succ_e EX_glu__L_e EX_ala__L_e
46 EX_succ_e EX_glu__L_e EX_ser__L_e
47 EX_succ_e EX_glu__L_e EX_nh4_e
48 EX_succ_e EX_ala__L_e EX_ser__L_e
49 EX_succ_e EX_ala__L_e EX_nh4_e
50 EX_succ_e EX_ser__L_e EX_nh4_e
51 EX_glyc_e EX_gln__L_e EX_glu__L_e
52 EX_glyc_e EX_gln__L_e EX_ala__L_e
53 EX_glyc_e EX_gln

In [37]:
combos=[('EX_gal_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_gal_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_gal_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_gal_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_gal_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_gal_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_gal_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_gal_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_gal_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_gal_e', 'EX_ser__L_e', 'EX_nh4_e'), ('EX_fru_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_fru_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_fru_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_fru_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_fru_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_fru_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_fru_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_fru_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_fru_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_fru_e', 'EX_ser__L_e', 'EX_nh4_e'), ('EX_ac_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_ac_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_ac_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_ac_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_ac_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_ac_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_ac_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_ac_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_ac_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_ac_e', 'EX_ser__L_e', 'EX_nh4_e'), ('EX_pyr_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_pyr_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_pyr_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_pyr_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_pyr_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_pyr_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_pyr_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_pyr_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_pyr_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_pyr_e', 'EX_ser__L_e', 'EX_nh4_e'), ('EX_succ_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_succ_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_succ_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_succ_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_succ_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_succ_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_succ_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_succ_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_succ_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_succ_e', 'EX_ser__L_e', 'EX_nh4_e'), ('EX_glyc_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_glyc_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_glyc_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_glyc_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_glyc_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_glyc_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_glyc_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_glyc_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_glyc_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_glyc_e', 'EX_ser__L_e', 'EX_nh4_e')]


In [11]:
#Basemedium, det universale som ligger i miljøet
def aapen(model_evolvex, threshold=-1e-9):
    open_ex = []
    for r in model_evolvex.reactions:
        if r.id.startswith("EX_") and r.lower_bound < threshold:
            open_ex.append((r.id, r.lower_bound, r.upper_bound))
    return sorted(open_ex)

open_uptakes = aapen(model_evolvex)
for x in open_uptakes[:100]:
    print(x)

('EX_ca2_e', -10000.0, 10000.0)
('EX_cl_e', -10000.0, 10000.0)
('EX_co2_e', -10000.0, 10000.0)
('EX_cobalt2_e', -10000.0, 10000.0)
('EX_cu2_e', -10000.0, 10000.0)
('EX_fe2_e', -10000.0, 10000.0)
('EX_fe3_e', -10000.0, 10000.0)
('EX_h2o_e', -10000.0, 10000.0)
('EX_h_e', -10000.0, 10000.0)
('EX_k_e', -10000.0, 10000.0)
('EX_mg2_e', -10000.0, 10000.0)
('EX_mn2_e', -10000.0, 10000.0)
('EX_mobd_e', -10000.0, 10000.0)
('EX_na1_e', -10000.0, 10000.0)
('EX_ni2_e', -10000.0, 10000.0)
('EX_o2_e', -10000.0, 10000.0)
('EX_pi_e', -10000.0, 10000.0)
('EX_sel_e', -10000.0, 10000.0)
('EX_slnt_e', -10000.0, 10000.0)
('EX_so4_e', -10000.0, 10000.0)
('EX_tungs_e', -10000.0, 10000.0)
('EX_zn2_e', -10000.0, 10000.0)


In [38]:
from EVOLVEX_SCORE import setBounds, createSecretion
#13.lager envs fra get_growth_supporting_niche1 og finner optimality
envs = []
for (c, n1, n2) in combos:   
    env = {
        c: "comp",
        n1: "comp",
        n2: "comp",
        "EX_glc__D_e": "inh",
        #"EX_lac__D_e": "inh"
    }
    envs.append(env)

components = sorted(set(rxn for combo in combos for rxn in combo))
setBounds(model_evolvex)
for ex in components:
    createSecretion(model_evolvex, ex)

for ex in components:
    r = model_evolvex.reactions.get_by_id(ex)
    r.lower_bound = 0
    r.upper_bound = 0

In [39]:
#14.sammenligner targets med alle mulige envs for å gi en evolvex score på beste miljø
strength, comp = enumerate_evolvex(model_evolvex, envs, targets_dict)

print(strength)

/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cplex/_internal/_pycplex.py:1632: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return _pycplex_platform.Pylolmat_to_CHBmat(lolmat, env_lp_ptr, py_row_col)


1/60 strength=-4.596855283078039 comp=['EX_gal_e', 'EX_gln__L_e', 'EX_glu__L_e']


/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cplex/_internal/_pycplex.py:1632: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return _pycplex_platform.Pylolmat_to_CHBmat(lolmat, env_lp_ptr, py_row_col)


2/60 strength=-6.339228752792016 comp=['EX_gal_e', 'EX_gln__L_e', 'EX_ala__L_e']


/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cplex/_internal/_pycplex.py:1632: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return _pycplex_platform.Pylolmat_to_CHBmat(lolmat, env_lp_ptr, py_row_col)


...

58/60 strength=-6.620162395609621 comp=['EX_glyc_e', 'EX_ala__L_e', 'EX_ser__L_e']


/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cplex/_internal/_pycplex.py:1632: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return _pycplex_platform.Pylolmat_to_CHBmat(lolmat, env_lp_ptr, py_row_col)


59/60 strength=-5.285185294726466 comp=['EX_glyc_e', 'EX_ala__L_e', 'EX_nh4_e']


/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cplex/_internal/_pycplex.py:1632: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return _pycplex_platform.Pylolmat_to_CHBmat(lolmat, env_lp_ptr, py_row_col)


60/60 strength=-6.414814842923985 comp=['EX_glyc_e', 'EX_ser__L_e', 'EX_nh4_e']
[-4.596855283078039, -6.339228752792016, -7.572120437765399, -6.722580810645497, -6.335483957149653, -7.561290352106762, -6.722580795944825, -9.31358970457045, -8.464516262273605, -9.690322693920942, -3.031074249017962, -4.5191491302662685, -5.6914374390407625, -4.434042650012854, -4.524890478937892, -5.668085219739564, -4.434042615317163, -7.159235781403006, -5.9234043272450245, -7.072340468027944, -1.0000000471569273, -2.0088039185057953, -3.0000001613246354, -1.0000000409392928, -2.0000001175892033, -3.000000161995727, -1.0000000479777498, -7.962790757794728, -6.81983084803439, -14.855561413174371, -1.0000001688655278, -2.000000071696019, -3.000000059903214, -1.0000001378759809, -2.009960801734459, -3.0000000634676454, -1.0000001350079692, -7.714659721491319, -6.562532313638517, -14.244444522012317, -1.3333816983873703e-07, -1.0018252253468418, -2.0000000768587807, -1.309623876011301e-07, -1.000000067360